# Federalist Papers: Hierarchische Clusteranalyse

Ergänzendes historisches Beispiel auf Grundlage der Exploration von Kai S. Kurono.

Vorbereitung: `python prepare_data.py --federalist`. Beide Notebooks verwenden denselben Parser in `stylometry/federalist.py`. Er entfernt Titel, Publikationsdaten, Autorenangaben, Anrede und PUBLIUS-Signatur samt nachfolgenden Endnoten. Von Aufsatz 70 bleibt die erste Fassung erhalten. Die Nummern stammen aus den Überschriften.

Die zwölf Aufsätze 49–58, 62 und 63 erhalten die Kategorie `Disputed`, die Aufsätze 18–20 die Kategorie `Joint`. Diese Kategorien dienen der Darstellung. Es handelt sich um eine explorative Visualisierung ohne Testklassifikation.

Die bereinigten Abbildungen ersetzen die früheren Darstellungen. Die PCA nutzt TF-IDF. Die HCA verwendet weiterhin Worthäufigkeiten mit deaktivierter IDF.

## 1. Bereinigte Texte und getrennte Metadaten

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from stylometry.federalist import load_federalist, COLORS

papers = pd.DataFrame(load_federalist())
display(papers.groupby('author').size().rename('Essays').to_frame())
assert papers.number.tolist() == list(range(1, 86))

## 2. Repräsentation und Berechnung

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

vectorizer = TfidfVectorizer(max_features=1000, use_idf=False, min_df=2, max_df=.9, stop_words='english')
vectors = vectorizer.fit_transform(papers.body).toarray()
# linkage expects a condensed distance vector, not a square distance matrix.
tree = linkage(pdist(vectors, metric='euclidean'), method='ward')
print('Feature matrix:', vectors.shape)

## 3. Darstellung und Interpretation

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(11, 15))
dendrogram(tree, labels=papers.number.tolist(), orientation='right', leaf_font_size=7, ax=ax)
authors_by_number = papers.set_index('number').author.to_dict()
for label in ax.get_yticklabels():
    label.set_color(COLORS[authors_by_number[int(label.get_text())]])
ax.legend(handles=[Line2D([0], [0], color=color, label=author) for author, color in COLORS.items()], loc='lower right')
ax.set(title='Federalist Papers: word frequencies and Ward linkage', xlabel='Ward linkage distance', ylabel='Essay number')
fig.tight_layout()
fig.savefig('federalist_hca.png', dpi=160)
plt.show()

Die Autorenfarben sind bekannte Metadaten. Sichtbare Gruppen beweisen keine zuverlässige Autorschaftserkennung. Die historischen Texte unterscheiden sich in Epoche und Genre von Reuters. Quellen: [Project Gutenberg](https://www.gutenberg.org/ebooks/18), [SciPy linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html).